# 04 — Heterogeneous Federated Learning Prototype V2

## Goal

This notebook tests a second heterogeneous federated-learning design for the two diabetes datasets.

Unlike V1, which restricted both clients to four manually shared features before knowledge distillation, V2 keeps each client's full usable local feature space. Each client has a **private encoder** that maps its preprocessed inputs into an 8-dimensional latent representation. Only the compatible **shared predictor** is federated.

Two controls are trained using the same architecture:

1. **Local-only models** — private encoder + private predictor, with no collaboration.
2. **Federated models** — private encoder + shared predictor, with repeated aggregation of predictor parameters.

This allows the effect of federated collaboration to be evaluated more fairly.


In [1]:
%pip install scikit-learn torch

import os, copy, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.max_columns", None)
print("PyTorch:", torch.__version__)


Note: you may need to restart the kernel to use updated packages.
PyTorch: 2.2.2


## 1. Load data and apply the same leakage exclusions as V1


In [2]:
DATASET1_PATH = "../data/diabetes_prediction_dataset.csv"
DATASET2_PATH = "../data/diabetes_dataset.csv"

df1 = pd.read_csv(DATASET1_PATH)
df2 = pd.read_csv(DATASET2_PATH)

TARGET1 = "diabetes"
TARGET2 = "diagnosed_diabetes"

# Same Dataset 2 leakage exclusions used in V1.
df2_model = df2.drop(
    columns=[c for c in ["diabetes_stage", "diabetes_risk_score"] if c in df2.columns]
).copy()

X1 = df1.drop(columns=[TARGET1])
y1 = df1[TARGET1].astype(int)

X2 = df2_model.drop(columns=[TARGET2])
y2 = df2_model[TARGET2].astype(int)

print("Dataset 1:", X1.shape)
print("Dataset 2:", X2.shape)
print("\nDataset 1 target distribution:")
print(y1.value_counts(normalize=True).sort_index().round(4))
print("\nDataset 2 target distribution:")
print(y2.value_counts(normalize=True).sort_index().round(4))


Dataset 1: (100000, 8)
Dataset 2: (100000, 28)

Dataset 1 target distribution:
diabetes
0    0.915
1    0.085
Name: proportion, dtype: float64

Dataset 2 target distribution:
diagnosed_diabetes
0    0.4
1    0.6
Name: proportion, dtype: float64


## 2. Train / validation / test splits

The original V1 80/20 train-test split is reproduced first using the same seed.  
The original 20% test set therefore remains unchanged.

Only the original 80% training portion is split again to create validation data.


In [3]:
# Reproduce the exact outer 80/20 split from V1.
X1_train_full, X1_test, y1_train_full, y1_test = train_test_split(
    X1, y1, test_size=0.20, random_state=SEED, stratify=y1
)
X2_train_full, X2_test, y2_train_full, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=SEED, stratify=y2
)

# 12.5% of the original 80% = 10% of the full dataset.
# Final proportions: 70% train / 10% validation / 20% test.
X1_train, X1_val, y1_train, y1_val = train_test_split(
    X1_train_full, y1_train_full,
    test_size=0.125, random_state=SEED, stratify=y1_train_full
)
X2_train, X2_val, y2_train, y2_val = train_test_split(
    X2_train_full, y2_train_full,
    test_size=0.125, random_state=SEED, stratify=y2_train_full
)

print("Dataset 1:", X1_train.shape, X1_val.shape, X1_test.shape)
print("Dataset 2:", X2_train.shape, X2_val.shape, X2_test.shape)


Dataset 1: (70000, 8) (10000, 8) (20000, 8)
Dataset 2: (70000, 28) (10000, 28) (20000, 28)


## 3. Client-specific preprocessing

Each client's preprocessing is fitted **only on its training data**.

- Numerical variables: median imputation + standardisation.
- Categorical variables: most-frequent imputation + one-hot encoding.

The resulting input dimensions are allowed to differ between clients.


In [4]:
def build_nn_preprocessor(X):
    categorical = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical = [c for c in X.columns if c not in categorical]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    return ColumnTransformer([
        ("num", numeric_pipe, numerical),
        ("cat", categorical_pipe, categorical)
    ])

prep1 = build_nn_preprocessor(X1_train)
prep2 = build_nn_preprocessor(X2_train)

A_train = prep1.fit_transform(X1_train).astype(np.float32)
A_val   = prep1.transform(X1_val).astype(np.float32)
A_test  = prep1.transform(X1_test).astype(np.float32)

B_train = prep2.fit_transform(X2_train).astype(np.float32)
B_val   = prep2.transform(X2_val).astype(np.float32)
B_test  = prep2.transform(X2_test).astype(np.float32)

print("Client A input dimension:", A_train.shape[1])
print("Client B input dimension:", B_train.shape[1])


Client A input dimension: 15
Client B input dimension: 46


## 4. V2 architecture

The encoders are client-specific and are **never averaged**.

Both encoders produce an 8-dimensional latent representation. The predictor therefore has an identical architecture on both clients and can be aggregated.


In [5]:
LATENT_DIM = 8

class ClientEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

class SharedPredictor(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, z):
        return self.net(z).squeeze(1)

class ClientModel(nn.Module):
    def __init__(self, encoder, predictor):
        super().__init__()
        self.encoder = encoder
        self.predictor = predictor

    def forward(self, x):
        return self.predictor(self.encoder(x))


In [6]:
def make_loader(X, y, batch_size=512, shuffle=True):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(np.asarray(y), dtype=torch.float32)
    return DataLoader(
        TensorDataset(X_t, y_t),
        batch_size=batch_size,
        shuffle=shuffle
    )

def train_model(model, X, y, epochs=5, lr=1e-3, batch_size=512, verbose=False):
    loader = make_loader(X, y, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        total_n = 0

        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(xb)
            total_n += len(xb)

        if verbose:
            print(f"Epoch {epoch + 1:02d} | loss={total_loss / total_n:.5f}")

    return model

def predict_prob(model, X):
    model.eval()
    X_t = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
        return torch.sigmoid(model(X_t)).cpu().numpy()

def evaluate_classifier(name, y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {
        "Model": name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, prob)
    }

def best_f1_threshold(y_true, prob):
    # Threshold selection uses validation data only.
    thresholds = np.linspace(0.05, 0.95, 181)
    scores = [
        f1_score(y_true, (prob >= t).astype(int), zero_division=0)
        for t in thresholds
    ]
    return float(thresholds[int(np.argmax(scores))])


## 5. Local-only control

These models use exactly the same encoder/predictor architecture as the federated experiment, but the clients never exchange or aggregate parameters.

This is the key control for testing whether collaboration itself adds value.


In [7]:
torch.manual_seed(SEED)

local_encoder_A = ClientEncoder(A_train.shape[1])
local_predictor_A = SharedPredictor()
local_model_A = ClientModel(local_encoder_A, local_predictor_A)

local_encoder_B = ClientEncoder(B_train.shape[1])
local_predictor_B = SharedPredictor()
local_model_B = ClientModel(local_encoder_B, local_predictor_B)

print("Training local-only Client A...")
train_model(local_model_A, A_train, y1_train, epochs=20, verbose=True)

print("\nTraining local-only Client B...")
train_model(local_model_B, B_train, y2_train, epochs=20, verbose=True)


Training local-only Client A...
Epoch 01 | loss=0.51848
Epoch 02 | loss=0.15458
Epoch 03 | loss=0.12208
Epoch 04 | loss=0.11332
Epoch 05 | loss=0.11030
Epoch 06 | loss=0.10772
Epoch 07 | loss=0.10505
Epoch 08 | loss=0.10189
Epoch 09 | loss=0.09838
Epoch 10 | loss=0.09588
Epoch 11 | loss=0.09383
Epoch 12 | loss=0.09192
Epoch 13 | loss=0.09035
Epoch 14 | loss=0.08923
Epoch 15 | loss=0.08798
Epoch 16 | loss=0.08723
Epoch 17 | loss=0.08585
Epoch 18 | loss=0.08509
Epoch 19 | loss=0.08433
Epoch 20 | loss=0.08375

Training local-only Client B...
Epoch 01 | loss=0.57979
Epoch 02 | loss=0.30738
Epoch 03 | loss=0.27922
Epoch 04 | loss=0.26248
Epoch 05 | loss=0.25050
Epoch 06 | loss=0.24327
Epoch 07 | loss=0.23861
Epoch 08 | loss=0.23545
Epoch 09 | loss=0.23324
Epoch 10 | loss=0.23140
Epoch 11 | loss=0.22989
Epoch 12 | loss=0.22886
Epoch 13 | loss=0.22801
Epoch 14 | loss=0.22704
Epoch 15 | loss=0.22663
Epoch 16 | loss=0.22625
Epoch 17 | loss=0.22545
Epoch 18 | loss=0.22471
Epoch 19 | loss=0.22419

ClientModel(
  (encoder): ClientEncoder(
    (net): Sequential(
      (0): Linear(in_features=46, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=8, bias=True)
      (3): ReLU()
    )
  )
  (predictor): SharedPredictor(
    (net): Sequential(
      (0): Linear(in_features=8, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=1, bias=True)
    )
  )
)

In [8]:
local_A_val_prob = predict_prob(local_model_A, A_val)
local_B_val_prob = predict_prob(local_model_B, B_val)

local_A_threshold = best_f1_threshold(y1_val, local_A_val_prob)
local_B_threshold = best_f1_threshold(y2_val, local_B_val_prob)

print("Local A validation threshold:", round(local_A_threshold, 3))
print("Local B validation threshold:", round(local_B_threshold, 3))


Local A validation threshold: 0.415
Local B validation threshold: 0.355


## 6. Federated V2 training

At the start of each round, both clients receive the same global predictor.

Each client then:
1. keeps its own private encoder,
2. trains locally using its own data,
3. updates its local copy of the shared predictor.

Only the predictor parameters are aggregated. The private encoders remain local.


In [11]:
def weighted_average_predictors(predictors, sample_counts):
    total = float(sum(sample_counts))
    avg_state = copy.deepcopy(predictors[0].state_dict())

    for key in avg_state:
        avg_state[key] = torch.zeros_like(avg_state[key])
        for predictor, n in zip(predictors, sample_counts):
            avg_state[key] += predictor.state_dict()[key] * (n / total)

    global_predictor = SharedPredictor()
    global_predictor.load_state_dict(avg_state)
    return global_predictor

def federated_train(
    X_A, y_A, X_B, y_B,
    rounds=5, local_epochs=3, lr=1e-3, batch_size=512
):
    torch.manual_seed(SEED)

    encoder_A = ClientEncoder(X_A.shape[1])
    encoder_B = ClientEncoder(X_B.shape[1])
    global_predictor = SharedPredictor()

    history = []

    for rnd in range(1, rounds + 1):
        # Each client starts the round from the same global predictor.
        predictor_A = copy.deepcopy(global_predictor)
        predictor_B = copy.deepcopy(global_predictor)

        model_A = ClientModel(encoder_A, predictor_A)
        model_B = ClientModel(encoder_B, predictor_B)

        train_model(
            model_A, X_A, y_A,
            epochs=local_epochs, lr=lr,
            batch_size=batch_size, verbose=False
        )
        train_model(
            model_B, X_B, y_B,
            epochs=local_epochs, lr=lr,
            batch_size=batch_size, verbose=False
        )

        # Encoders have been updated locally through model_A/model_B.
        encoder_A = model_A.encoder
        encoder_B = model_B.encoder

        # Only compatible predictor parameters are federated.
        global_predictor = weighted_average_predictors(
            [model_A.predictor, model_B.predictor],
            [len(X_A), len(X_B)]
        )

        fed_A = ClientModel(encoder_A, copy.deepcopy(global_predictor))
        fed_B = ClientModel(encoder_B, copy.deepcopy(global_predictor))

        history.append({"Round": rnd})
        print(f"Completed federated round {rnd}/{rounds}")

    final_A = ClientModel(encoder_A, copy.deepcopy(global_predictor))
    final_B = ClientModel(encoder_B, copy.deepcopy(global_predictor))

    return final_A, final_B, global_predictor, history

fed_model_A, fed_model_B, global_predictor_v2, fed_history = federated_train(
    A_train, y1_train,
    B_train, y2_train,
    rounds=5,
    local_epochs=3
)


Completed federated round 1/5
Completed federated round 2/5
Completed federated round 3/5
Completed federated round 4/5
Completed federated round 5/5


## 7. Validation-based threshold selection

Thresholds are selected separately for each dataset using **validation data only**.

The test sets remain untouched until the final evaluation below.


In [12]:
fed_A_val_prob = predict_prob(fed_model_A, A_val)
fed_B_val_prob = predict_prob(fed_model_B, B_val)

fed_A_threshold = best_f1_threshold(y1_val, fed_A_val_prob)
fed_B_threshold = best_f1_threshold(y2_val, fed_B_val_prob)

print("Federated A validation threshold:", round(fed_A_threshold, 3))
print("Federated B validation threshold:", round(fed_B_threshold, 3))


Federated A validation threshold: 0.87
Federated B validation threshold: 0.395


## 8. Final held-out test evaluation

Two versions are reported:

- **Threshold = 0.5**, for continuity with V1.
- **Validation-calibrated threshold**, selected before looking at the test labels.

ROC-AUC is threshold-independent.


In [13]:
results_v2 = []

local_A_test_prob = predict_prob(local_model_A, A_test)
local_B_test_prob = predict_prob(local_model_B, B_test)
fed_A_test_prob = predict_prob(fed_model_A, A_test)
fed_B_test_prob = predict_prob(fed_model_B, B_test)

# Default 0.5 threshold
results_v2.append(evaluate_classifier(
    "V2 Local A - Dataset 1 (0.5)", y1_test, local_A_test_prob, 0.5
))
results_v2.append(evaluate_classifier(
    "V2 Local B - Dataset 2 (0.5)", y2_test, local_B_test_prob, 0.5
))
results_v2.append(evaluate_classifier(
    "V2 Federated A - Dataset 1 (0.5)", y1_test, fed_A_test_prob, 0.5
))
results_v2.append(evaluate_classifier(
    "V2 Federated B - Dataset 2 (0.5)", y2_test, fed_B_test_prob, 0.5
))

# Validation-calibrated thresholds
results_v2.append(evaluate_classifier(
    "V2 Local A - Dataset 1 (calibrated)",
    y1_test, local_A_test_prob, local_A_threshold
))
results_v2.append(evaluate_classifier(
    "V2 Local B - Dataset 2 (calibrated)",
    y2_test, local_B_test_prob, local_B_threshold
))
results_v2.append(evaluate_classifier(
    "V2 Federated A - Dataset 1 (calibrated)",
    y1_test, fed_A_test_prob, fed_A_threshold
))
results_v2.append(evaluate_classifier(
    "V2 Federated B - Dataset 2 (calibrated)",
    y2_test, fed_B_test_prob, fed_B_threshold
))

results_v2_df = pd.DataFrame(results_v2)

for c in ["Threshold", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]:
    results_v2_df[c] = results_v2_df[c].round(4)

results_v2_df


,Model,Threshold,Accuracy,Precision,Recall,F1,ROC-AUC
0,V2 Local A - Dataset 1 (0.5),0.500,0.9706,0.9595,0.6835,0.7984,0.9758
1,V2 Local B - Dataset 2 (0.5),0.500,0.9100,0.9902,0.8585,0.9197,0.9416
2,V2 Federated A - Dataset 1 (0.5),0.500,0.9256,0.5415,0.8182,0.6517,0.9636
3,V2 Federated B - Dataset 2 (0.5),0.500,0.9113,0.9938,0.8575,0.9206,0.9421
4,V2 Local A - Dataset 1 (calibrated),0.415,0.9698,0.9329,0.6953,0.7968,0.9758
5,V2 Local B - Dataset 2 (calibrated),0.355,0.9096,0.9784,0.8684,0.9201,0.9416
6,V2 Federated A - Dataset 1 (calibrated),0.870,0.9678,0.9483,0.6576,0.7767,0.9636
7,V2 Federated B - Dataset 2 (calibrated),0.395,0.9114,0.9885,0.8622,0.9211,0.9421


## 9. Collaboration-effect summary

This table directly addresses the question:

> Does the federated version improve or degrade performance relative to the same architecture trained locally?

Positive deltas indicate improvement from collaboration; negative deltas indicate degradation.


In [14]:
def collaboration_delta(local_row, fed_row, dataset):
    metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
    return {
        "Dataset": dataset,
        **{
            f"Delta {m}": float(fed_row[m]) - float(local_row[m])
            for m in metrics
        }
    }

# Use calibrated rows for the threshold-dependent metrics.
local_A_row = results_v2[-4]
local_B_row = results_v2[-3]
fed_A_row   = results_v2[-2]
fed_B_row   = results_v2[-1]

delta_df = pd.DataFrame([
    collaboration_delta(local_A_row, fed_A_row, "Dataset 1"),
    collaboration_delta(local_B_row, fed_B_row, "Dataset 2")
])

for c in delta_df.columns[1:]:
    delta_df[c] = delta_df[c].round(4)

delta_df


,Dataset,Delta Accuracy,Delta Precision,Delta Recall,Delta F1,Delta ROC-AUC
0,Dataset 1,-0.0020,0.0153,-0.0376,-0.0201,-0.0122
1,Dataset 2,0.0018,0.0101,-0.0062,0.0009,0.0005


## 10. Save V2 artefacts


In [15]:
os.makedirs("../models", exist_ok=True)

torch.save(
    global_predictor_v2.state_dict(),
    "../models/heterogeneous_global_predictor_v2.pt"
)

results_v2_df.to_csv(
    "../models/prototype_v2_metrics.csv",
    index=False
)

delta_df.to_csv(
    "../models/prototype_v2_collaboration_delta.csv",
    index=False
)

print("Saved ../models/heterogeneous_global_predictor_v2.pt")
print("Saved ../models/prototype_v2_metrics.csv")
print("Saved ../models/prototype_v2_collaboration_delta.csv")


Saved ../models/heterogeneous_global_predictor_v2.pt
Saved ../models/prototype_v2_metrics.csv
Saved ../models/prototype_v2_collaboration_delta.csv
